In [1]:
import json
from pathlib import Path
from datetime import datetime
from funciones_auxiliares import extract_submissions, extract_comments_for_submissions, stream_zst_file

# --- MIS 3 SUBREDDITS ---
mis_subreddits = ["travel", "MealPrepSunday", "Fitness"] 

# Parámetros solicitados en el enunciado 
HILOS_POR_SUBREDDIT = 40
COMENTARIOS_POR_HILO = 25

for sub in mis_subreddits:
    print(f"\n{'='*40}")
    print(f"Iniciando extracción de: r/{sub}")
    print(f"{'='*40}")

    # 1. Extraer los 40 hilos (submissions)
    # Filtramos hilos que tengan al menos 25 comentarios para asegurar el corpus
    mis_submissions = extract_submissions(
        "datos/RS_2025.zst", 
        sub, 
        n_submissions=HILOS_POR_SUBREDDIT, 
        min_comments=COMENTARIOS_POR_HILO
    )

    if mis_submissions:
        # 2. Extraer los 25 comentarios para cada uno de esos hilos
        extract_comments_for_submissions(
            "datos/RC_2025.zst", 
            mis_submissions, 
            num_comments=COMENTARIOS_POR_HILO
        )

        # 3. Estructurar el resultado según el ejemplo del enunciado [cite: 42-47]
        resultado_final = {
            "subreddit": sub,
            "extraction_date": datetime.now().isoformat(),
            "num_submissions": len(mis_submissions),
            "total_comments": sum(len(s['comments']) for s in mis_submissions),
            "submissions": mis_submissions
        }

        # 4. Guardar en JSON individual [cite: 40, 41]
        nombre_archivo = f"ejemplo_subreddit_{sub}.json"
        with open(nombre_archivo, 'w', encoding='utf-8') as f:
            json.dump(resultado_final, f, ensure_ascii=False, indent=2)
        
        print(f"Archivo '{nombre_archivo}' generado con éxito.")
    else:
        print(f"No se encontraron suficientes hilos en r/{sub}")


🚀 Iniciando extracción de: r/travel
🔍 Buscando submissions en r/travel con ≥25 comentarios...
  ✓ [1/40] (128 comments) Longest flight itinerary you have taken (inc. tran...
  ✓ [2/40] (60 comments) Is Is Argentina cheap or expensive right now for t...
  ✓ [3/40] (122 comments) How many different countries did you visit in 2024...
  ✓ [4/40] (37 comments) My first trip to Armenia part 2...
  ✓ [5/40] (28 comments) Which country/city did you end up loving more than...
  ✓ [6/40] (36 comments) Hello everyone I will visit Japan and I am hesitan...
  ✓ [7/40] (25 comments) Best way to travel to NYC...
  ✓ [8/40] (39 comments) Travel to Mexico with one parent...
  ✓ [9/40] (32 comments) Southeast Asia in a month...
  ✓ [10/40] (28 comments) Belize with kids ...
  ✓ [11/40] (88 comments) I got robbed at Brussels on New Years eve :(...
  ✓ [12/40] (32 comments) Where to go in Maine?...
  ✓ [13/40] (43 comments) Advice on Spain ...
  ✓ [14/40] (75 comments) Two weeks in Ukraine! Kyiv, Kharkiv

In [2]:
import json
from datetime import datetime

# Lista de los archivos que generaste anteriormente
archivos_json = ["ejemplo_subreddit_travel.json", "ejemplo_subreddit_MealPrepSunday.json", "ejemplo_subreddit_Fitness.json"]

for archivo in archivos_json:
    try:
        with open(archivo, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print(f"\n{'='*50}")
        print(f"📅 ANÁLISIS TEMPORAL: r/{data['subreddit']}")
        print(f"{'='*50}")

        # Extraer fechas de creación de las submissions (convertidas de UTC a datetime)
        # created_utc viene en los datos originales del volcado [cite: 58]
        fechas = [datetime.fromtimestamp(s['created_utc']) for s in data['submissions']]
        
        if fechas:
            fechas_ordenadas = sorted(fechas)
            primera = fechas_ordenadas[0]
            ultima = fechas_ordenadas[-1]
            rango_dias = (ultima - primera).days

            print(f"🔹 Primera publicación: {primera.strftime('%Y-%m-%d %H:%M')}")
            print(f"🔹 Última publicación:  {ultima.strftime('%Y-%m-%d %H:%M')}")
            print(f"🔹 Amplitud temporal:    {rango_dias} días")

            # Contar cuántas hay por día para ver la densidad
            dias = [f.strftime('%Y-%m-%d') for f in fechas]
            conteo_dias = {dia: dias.count(dia) for dia in set(dias)}
            
            print("\n📊 Distribución por días (Primeros 5 días detectados):")
            for dia in sorted(conteo_dias.keys())[:5]:
                print(f"   - {dia}: {conteo_dias[dia]} hilos")
            
            if rango_dias < 1:
                print("\n⚠️ ALERTA: Todos los hilos son del mismo día. Considera saltar registros en la extracción.")
            else:
                print("\n✅ El corpus presenta variedad temporal.")
        else:
            print("❌ No se encontraron fechas en las submissions.")

    except FileNotFoundError:
        print(f"⚠️ No se encontró el archivo: {archivo}")


📅 ANÁLISIS TEMPORAL: r/travel
🔹 Primera publicación: 2025-01-01 03:49
🔹 Última publicación:  2025-01-02 23:44
🔹 Amplitud temporal:    1 días

📊 Distribución por días (Primeros 5 días detectados):
   - 2025-01-01: 19 hilos
   - 2025-01-02: 21 hilos

✅ El corpus presenta variedad temporal.

📅 ANÁLISIS TEMPORAL: r/MealPrepSunday
🔹 Primera publicación: 2025-01-02 05:43
🔹 Última publicación:  2025-01-21 13:39
🔹 Amplitud temporal:    19 días

📊 Distribución por días (Primeros 5 días detectados):
   - 2025-01-02: 1 hilos
   - 2025-01-03: 1 hilos
   - 2025-01-04: 3 hilos
   - 2025-01-05: 4 hilos
   - 2025-01-06: 5 hilos

✅ El corpus presenta variedad temporal.

📅 ANÁLISIS TEMPORAL: r/Fitness
🔹 Primera publicación: 2025-01-01 11:00
🔹 Última publicación:  2025-01-22 11:00
🔹 Amplitud temporal:    21 días

📊 Distribución por días (Primeros 5 días detectados):
   - 2025-01-01: 2 hilos
   - 2025-01-02: 1 hilos
   - 2025-01-03: 2 hilos
   - 2025-01-04: 2 hilos
   - 2025-01-05: 2 hilos

✅ El corpus p

In [3]:
import json
import re


def analizar_calidad(texto):
    # Patrones para detectar URLs y Emails
    tiene_url = bool(re.search(r'https?://\S+|www\.\S+', texto))
    tiene_email = bool(re.search(r'\S+@\S+\.\S+', texto))
    longitud = len(texto.split()) # Contamos palabras
    return longitud, tiene_url, tiene_email

for archivo in archivos_json:
    with open(archivo, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    total_comentarios = 0
    cortos = 0 # Menos de 5 palabras
    solo_links = 0
    con_email = 0
    
    for submission in data['submissions']:
        for comment in submission.get('comments', []):
            total_comentarios += 1
            cuerpo = comment.get('body', '')
            
            n_palabras, has_url, has_email = analizar_calidad(cuerpo)
            
            if n_palabras < 5:
                cortos += 1
            if has_url and n_palabras < 3: # Muy corto y con URL suele ser solo spam/link
                solo_links += 1
            if has_email:
                con_email += 1

    print(f"\n{'='*50}")
    print(f"🔍 CALIDAD DEL TEXTO: r/{data['subreddit']}")
    print(f"{'='*50}")
    print(f"✅ Total analizados: {total_comentarios}")
    print(f"⚠️ Comentarios muy cortos (< 5 palabras): {cortos} ({cortos/total_comentarios*100:.1f}%)")
    print(f"🔗 Comentarios que son casi solo URLs: {solo_links}")
    print(f"📧 Comentarios con emails: {con_email}")
    
    if cortos / total_comentarios > 0.2:
        print("💡 Sugerencia: El corpus tiene mucho 'ruido' (mensajes cortos). Deberías filtrar en el siguiente paso.")


🔍 CALIDAD DEL TEXTO: r/travel
✅ Total analizados: 1000
⚠️ Comentarios muy cortos (< 5 palabras): 101 (10.1%)
🔗 Comentarios que son casi solo URLs: 0
📧 Comentarios con emails: 0

🔍 CALIDAD DEL TEXTO: r/MealPrepSunday
✅ Total analizados: 1000
⚠️ Comentarios muy cortos (< 5 palabras): 109 (10.9%)
🔗 Comentarios que son casi solo URLs: 1
📧 Comentarios con emails: 0

🔍 CALIDAD DEL TEXTO: r/Fitness
✅ Total analizados: 1000
⚠️ Comentarios muy cortos (< 5 palabras): 194 (19.4%)
🔗 Comentarios que son casi solo URLs: 4
📧 Comentarios con emails: 0
